# Projeto de Predição de Preços de Carros — ML
Este notebook performa diversos treinamentos com modelos de regressão  utilizando um dataset de carros usados da CarDekho, buscando comparar diferentes formas de se prever um novo resultado.

# Importando o Dataset e as Bibliotecas

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nehalbirla/vehicle-dataset-from-cardekho")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'vehicle-dataset-from-cardekho' dataset.
Path to dataset files: /kaggle/input/vehicle-dataset-from-cardekho


In [3]:
df = pd.read_csv("/kaggle/input/vehicle-dataset-from-cardekho/CAR DETAILS FROM CAR DEKHO.csv")
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


Como vimos na análise exploratória, este dataset tem linhas repetidas, então vamos removê-las antes de qualquer coisa.

In [4]:
df.drop_duplicates(inplace=True)

# Feature Engineering
Durante a etapa de feature engineering, criei uma nova feature `car_year` baseada na coluna `year` para facilitar a análise pela idade do carro ao invés do ano de lançamento. E também foi criada a variável `brand` a partir da coluna `name`, extraindo a marca de cada veículo, já que a marca possui forte influência no preço dos carros.

In [5]:
df['car_year'] = 2025 - df['year']

df['brand'] = df['name'].str.split().str[0]

# Tratamento de Outliers
Foi realizada uma análise de outliers no target utilizando estatísticas descritivas. Como o dataset apresentava valores extremamente altos que poderiam impactar negativamente os modelos, foi aplicado um filtro por percentis para reduzir a influência de valores extremos e melhorar a capacidade de generalização dos algoritmos.


In [6]:
df['selling_price'].describe()

,selling_price
count,3.577000e+03
mean,4.739125e+05
std,5.093018e+05
min,2.000000e+04
25%,2.000000e+05
50%,3.500000e+05
75%,6.000000e+05
max,8.900000e+06


In [7]:
low = df['selling_price'].quantile(0.01)
high = df['selling_price'].quantile(0.99)

df_filtered = df[
    (df['selling_price'] >= low)
    &
    (df['selling_price'] <= high)
]

df_filtered

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,car_year,brand
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner,18,Maruti
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner,18,Maruti
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner,13,Hyundai
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner,8,Datsun
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner,11,Honda
...,...,...,...,...,...,...,...,...,...,...
4335,Hyundai i20 Magna 1.4 CRDi (Diesel),2014,409999,80000,Diesel,Individual,Manual,Second Owner,11,Hyundai
4336,Hyundai i20 Magna 1.4 CRDi,2014,409999,80000,Diesel,Individual,Manual,Second Owner,11,Hyundai
4337,Maruti 800 AC BSIII,2009,110000,83000,Petrol,Individual,Manual,Second Owner,16,Maruti
4338,Hyundai Creta 1.6 CRDi SX Option,2016,865000,90000,Diesel,Individual,Manual,First Owner,9,Hyundai


# Pré-processamento de Dados
Na etapa de preprocessing, as variáveis categóricas foram transformadas utilizando One Hot Encoding para permitir que os modelos trabalhassem corretamente com dados textuais. Também foram aplicadas transformações e escalonamentos necessários para preparar os dados para o treinamento dos algoritmos de regressão.

In [8]:
X = df_filtered.drop(['selling_price', 'name', 'year'], axis=1)
y = df_filtered['selling_price']

In [9]:
categorical_features = ['fuel', 'seller_type', 'transmission', 'owner', 'brand']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

X_transformed  = preprocessor.fit_transform(X)

Aqui estamos separando os dados em treino e teste. Escolhi usar 20% dos dados para teste, já que este é o padrão para datasets menores. O random_state=42 é apenas para garantir que o dataset seja dividido da mesma forma toda vez que esta célula é executada.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X_transformed, y, test_size=0.2, random_state=42)

E para finalizar, fizemos novas variáveis com os dados padronizados. Será muito útil para modelos como SVR, que trabalha com escalas e necessita de precisão em seus valores.

In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train_scaled = scaler.fit_transform(y_train.values.reshape(-1,1))
y_test_scaled = scaler.transform(y_test.values.reshape(-1,1))

# Linear Regression



In [12]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

LinearRegression()

In [13]:
y_pred_lr = lr_model.predict(X_test_scaled)

lr_r2 = r2_score(y_test, y_pred_lr)
lr_rmse = root_mean_squared_error(y_test, y_pred_lr)

# Support Vector Regression (SVR)


In [14]:
svr_model = SVR(kernel='rbf')
svr_model.fit(X_train_scaled, y_train_scaled.ravel())

SVR()

In [15]:
y_pred_svr = svr_model.predict(X_test_scaled)
y_pred_svr = scaler.inverse_transform(y_pred_svr.reshape(-1,1))

svr_r2 = r2_score(y_test, y_pred_svr)
svr_rmse = root_mean_squared_error(y_test, y_pred_svr)

# Decision Tree

In [16]:
params_tree = {
    'max_depth': [3,5,10,None],
    'min_samples_split': [2,5,10],
    'min_samples_leaf': [1,2,4]
}

grid_tree = GridSearchCV(
    DecisionTreeRegressor(),
    params_tree,
    cv=5,
    scoring='r2'
)

grid_tree.fit(X_train, y_train)

print(grid_tree.best_params_)

{'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 10}


In [17]:
y_pred_tree = grid_tree.predict(X_test)

tree_r2 = r2_score(y_test, y_pred_tree)
tree_rmse = root_mean_squared_error(y_test, y_pred_tree)

# Random Forest

In [18]:
params_forest = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_leaf': [1, 2, 4]
}

grid_forest = GridSearchCV(
    RandomForestRegressor(),
    params_forest,
    cv=5,
    scoring='r2'
)

grid_forest.fit(X_train, y_train)

print(grid_forest.best_params_)

{'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 100}


In [19]:
y_pred_forest = grid_forest.predict(X_test)

forest_r2 = r2_score(y_test, y_pred_forest)
forest_rmse = root_mean_squared_error(y_test, y_pred_forest)

# Comparação Final

In [23]:
df['selling_price'].median()

350000.0

In [21]:
results = pd.DataFrame({
    'Modelo': ['Linear Regression', 'Support Vector Regression', 'Decision Tree', 'Random Forest'],
    'R2': [lr_r2, svr_r2, tree_r2, forest_r2 ],
    'RMSE': [lr_rmse, svr_rmse, tree_rmse, forest_rmse]
})
results.sort_values(by='R2', ascending=False)

,Modelo,R2,RMSE
1,Support Vector Regression,0.733528,182312.720718
3,Random Forest,0.726531,184690.630921
0,Linear Regression,0.684658,198326.807621
2,Decision Tree,0.669549,203022.686933


Podemos interpretar que o modelo que melhor performou com este dataset foi o Support Vector Regression, alcançando um maior valor de R² e menor valor de RMSE. Respectivamente, isso significa que o modelo explica ~73% da variação dos preços, demonstrando uma boa capacidade de capturar padrões existentes no dataset, e já o menor RMSE demonstra que o modelo produziu previsões com menor desvio médio em relação aos valores reais.

A combinação de alto R² e baixo RMSE sugere que o modelo possui melhor capacidade de generalização para este problema, equilibrando boa explicação dos dados e menor erro de previsão.